## K3s Virtual Cluster

Ein k3s Virtual Cluster stellt eine isolierte Kubernetes-Umgebung innerhalb eines bestehenden Host-Clusters bereit. 

Dabei läuft eine eigene Control Plane mit separater API, eigenen Nodes und eigener Cluster-Konfiguration, während die zugrunde liegende Infrastruktur des Host-Clusters weiterverwendet wird.

Dieses Setup eignet sich besonders für Tests, Schulungen und mandantenfähige Umgebungen, da Cluster schnell erstellt, gelöscht und voneinander getrennt betrieben werden können.

---

Zuerst Anzahl Filehandles etc. erhöhen

In [ ]:
%%bash
sudo sysctl -w fs.inotify.max_user_instances=8192
sudo sysctl -w fs.inotify.max_user_watches=1048576
sudo sysctl -w fs.file-max=2097152
cat <<EOF | sudo tee /etc/sysctl.d/99-k3k-inotify.conf
fs.inotify.max_user_instances = 8192
fs.inotify.max_user_watches = 1048576
fs.file-max = 2097152
EOF

sudo sysctl --system

Dann Installieren wie den Operator

In [ ]:
%%bash
helm repo add k3k https://rancher.github.io/k3k
helm repo update

In [ ]:
%%bash
helm install k3k k3k/k3k --namespace k3k-system --create-namespace

und das CLI

In [ ]:
%%bash
wget -qO k3kcli https://github.com/rancher/k3k/releases/download/v1.1.0/k3kcli-linux-amd64 && \
  chmod +x k3kcli && \
  sudo mv k3kcli /usr/local/bin

---

### Cluster

Cluster erstellen

In [ ]:
%%bash
kubectl create namespace k3k-test
kubectl label namespace k3k-test \
  pod-security.kubernetes.io/enforce=privileged \
  pod-security.kubernetes.io/audit=privileged \
  pod-security.kubernetes.io/warn=privileged \
  --overwrite

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: k3k.io/v1beta1
kind: Cluster
metadata:
  name: test
  namespace: k3k-test
spec:
  mode: virtual
  servers: 1
  agents: 2

  version: v1.35.5-k3s1

  serverArgs:
    - "--snapshotter=native"
  agentArgs:
    - "--snapshotter=native"

  persistence:
    type: dynamic
    storageRequestSize: 20Gi

  expose:
    loadBalancer: {}

  sync:
    services:
      enabled: true
    configMaps:
      enabled: true
    secrets:
      enabled: true
EOF

Pro Node sollte ein Pod laufen

In [ ]:
%%bash
kubectl get pods -n k3k-test

Wir brauchen die KUBECONFIG 

In [ ]:
%%bash
kubectl get secret k3k-test-kubeconfig -n k3k-test -o jsonpath='{.data.kubeconfig\.yaml}' | base64 -d | tee ~/data/k3s-test-config

### Testen

Schauen was läuft

In [ ]:
%%bash
kubectl --kubeconfig ~/data/k3s-test-config get nodes -o wide
kubectl --kubeconfig ~/data/k3s-test-config get pods -A -o wide

---

### Aufräumen

In [ ]:
%%bash
kubectl delete cluster/test -n k3k-test --ignore-not-found
kubectl delete namespace k3k-test --ignore-not-found

In [ ]:
%%bash
helm uninstall k3k -n k3k-system
kubectl delete ns k3k-system || true